# Lab 4 — The same design principles with LangChain middleware (EP06)

In Labs 1–3 we wrote everything ourselves: a clear tool · retry · answering from a cache · a trace · the date in the prompt
This lab does the same with LangChain's `create_agent` and **middleware** — pieces that plug in around a model call or a tool call

| We wrote it ourselves in | The LangChain piece |
|---|---|
| Lab 1: a dict tool schema | `@tool` + docstring |
| Lab 1: `with_retry` | `ToolRetryMiddleware` |
| Lab 1: `run_degrade` + `final_answer` | `@wrap_tool_call` + `@after_agent` |
| Lab 2: `SYSTEM += date` | `@dynamic_prompt` |
| Lab 2: `trace` + `show_trace` | `agent.stream(stream_mode="updates")` |
| Lab 3: `max_steps` / a call count that varies | `ModelCallLimitMiddleware` |

The paper library and the failing search API are **simulated data written for teaching**; every model answer comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.3 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs well under one cent) |
| Packages | `langchain`, `langgraph`, `langchain-openai`, `langchain-core`, `python-dotenv` (pinned to the same versions as EP05 Lab 5) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

How to set up the key, and why we chose this model, is in Lab 1

In [ ]:
%pip install -q "langchain==1.4.7" "langgraph==1.2.16" "langchain-openai==1.7.2" "langchain-core==1.7.0" python-dotenv

## 1. The model + paper library + a search API that can fail

`ChatOpenAI` pointing to OpenRouter (Gemma 4 26B pinned to DeepInfra) · `PAPERS`, `find_papers`, `SearchTimeout` and `down` come from Labs 1–2

In [ ]:
import json, os, datetime
from langchain_openai import ChatOpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

In [ ]:
llm = ChatOpenAI(model="google/gemma-4-26b-a4b-it", base_url="https://openrouter.ai/api/v1", api_key=load_key(),
                 extra_body={"reasoning": {"enabled": False},
                             "provider": {"order": ["deepinfra/fp8"], "allow_fallbacks": False}})

In [ ]:
PAPERS = [
    {"id": "P01", "title": "Informer: efficient transformer for long sequence time-series forecasting", "year": 2021},
    {"id": "P02", "title": "Autoformer: decomposition transformers for long-term series forecasting", "year": 2021},
    {"id": "P03", "title": "FEDformer: frequency enhanced decomposed transformer for time series", "year": 2022},
    {"id": "P04", "title": "Are transformers effective for time series forecasting?", "year": 2023},
    {"id": "P05", "title": "PatchTST: a time series is worth 64 words", "year": 2023},
    {"id": "P06", "title": "iTransformer: inverted transformers for time series forecasting", "year": 2024},
    {"id": "P07", "title": "TimesFM: a decoder-only foundation model for time-series forecasting", "year": 2024},
    {"id": "P08", "title": "Chronos: learning the language of time series", "year": 2024},
    {"id": "P09", "title": "Attention is all you need", "year": 2017},
    {"id": "P10", "title": "BERT: pre-training of deep bidirectional transformers", "year": 2019},
    {"id": "P11", "title": "Temporal fusion transformers for interpretable time series forecasting", "year": 2019},
    {"id": "P12", "title": "Moirai: unified training of universal time series forecasting transformers", "year": 2024},
]
STOP = {"with", "from", "paper", "papers", "model", "models", "using", "published", "since", "research", "about", "learning"}

def find_papers(query, year_from=0, library=PAPERS):
    words = [w for w in query.lower().replace("-", " ").split() if len(w) > 3 and w not in STOP]
    return [p for p in library
            if p["year"] >= year_from and any(w in p["title"].lower().replace("-", " ") for w in words)]

class SearchTimeout(Exception):
    pass

down = {"fails_left": 0}

## 2. ③ A good tool: `@tool` + docstring

Function name → tool name · type hints → argument types · docstring → description (`parse_docstring=True` reads the `Args:` section as each argument's description)

In [ ]:
from langchain_core.tools import tool

@tool(parse_docstring=True)
def search_papers(query: str, year_from: int) -> list[dict]:
    """Search academic papers in the Cogito Learn library by topic. Returns a list of {id, title, year}; returns [] if none match.

    Args:
        query: Topic in English, e.g. 'time series transformer'
        year_from: Earliest publication year wanted (CE), e.g. 2023 = from 2023 onward
    """
    if down["fails_left"] > 0:
        down["fails_left"] -= 1
        print("   ⚠️ search API timeout")
        raise SearchTimeout("search API did not respond within 10 seconds")
    return find_papers(query, year_from)

print(json.dumps(search_papers.args, ensure_ascii=False, indent=1))

## 3. ② A trace: `stream_mode="updates"`

`create_agent` builds a LangGraph graph for us · `stream` emits every node's output step by step — the model node, the tool node, and some middleware's own nodes

In [ ]:
def ask(agent, question):
    for update in agent.stream({"messages": [{"role": "user", "content": question}]}, stream_mode="updates"):
        for node, value in update.items():
            for m in (value or {}).get("messages", []):
                calls = getattr(m, "tool_calls", None)
                if calls:
                    what = f"🔧 asks for {[(c['name'], c['args']) for c in calls]}"
                elif m.type == "tool":
                    what = f"📄 result: {str(m.content)[:90]}"
                else:
                    what = f"💬 answers: {str(m.content)[:90]!r}"
                print(f"[{node}] {what}")
    return m.content

## 4. `create_agent` + `@dynamic_prompt`

In [ ]:
from langchain.agents import create_agent

SYSTEM = "You are the Cogito Learn paper-search assistant. Always use tools to get real data; never answer from memory."
RECENT = "Find papers on transformers for time series from the last 2 years"

plain = create_agent(llm, tools=[search_papers], system_prompt=SYSTEM)
answer = ask(plain, RECENT)

In [ ]:
from langchain.agents.middleware import dynamic_prompt

@dynamic_prompt
def with_today(request):
    return SYSTEM + " Today is " + str(datetime.date.today()) + "."

dated = create_agent(llm, tools=[search_papers], middleware=[with_today])
answer = ask(dated, RECENT)

## 5. ④ Plan for failure: retry → cache → attach a warning

Middleware that wraps tool calls nests in list order — **the first one is outermost**:
`cache_fallback` ( `ToolRetryMiddleware` ( tool ) ) → retry first; if it's still down, fall back to the cache

In [ ]:
from langchain.agents.middleware import ToolRetryMiddleware

retry = ToolRetryMiddleware(retry_on=(SearchTimeout,), max_retries=2,
                            initial_delay=1, backoff_factor=2, jitter=False, on_failure="error")

In [ ]:
from langchain.agents.middleware import wrap_tool_call, after_agent
from langchain_core.messages import ToolMessage, AIMessage

SNAPSHOT_DATE = "2024-01-15"
SNAPSHOT = [p for p in PAPERS if p["year"] <= 2023]
notes = []

@wrap_tool_call
def cache_fallback(request, handler):
    try:
        return handler(request)
    except SearchTimeout:
        args = request.tool_call["args"]
        notes.append(f"⚠️ search is unavailable; these results come from the library copy of {SNAPSHOT_DATE} and may miss newer papers")
        papers = find_papers(args["query"], args["year_from"], library=SNAPSHOT)
        return ToolMessage(json.dumps({"source": f"library copy of {SNAPSHOT_DATE} — may be out of date", "papers": papers},
                                      ensure_ascii=False), tool_call_id=request.tool_call["id"])

@after_agent
def add_notes(state, runtime):
    if notes:
        last = state["messages"][-1]
        return {"messages": [AIMessage("\n".join(notes) + "\n" + last.content, id=last.id)]}

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware

agent = create_agent(llm, tools=[search_papers],
                     middleware=[with_today, cache_fallback, retry, add_notes,
                                 ModelCallLimitMiddleware(run_limit=6)])

In [ ]:
Q2023 = "Find papers on transformers for time series published in 2023 or later"

down["fails_left"] = 2
answer = ask(agent, Q2023)

In [ ]:
notes.clear()
down["fails_left"] = 99
answer = ask(agent, Q2023)
print(answer)

## Summary

| Principle | Written ourselves (Labs 1–3) | LangChain |
|---|---|---|
| ③ good tools | a dict schema | `@tool(parse_docstring=True)` |
| ② trace | `trace` + `show_trace` | `stream_mode="updates"` — middleware that adds a message shows up as its own node (`add_notes.after_agent`) |
| what the model must know | `SYSTEM += date` | `@dynamic_prompt` |
| ④ retry | `with_retry` | `ToolRetryMiddleware(retry_on=…, on_failure="error")` |
| ④ graceful degradation | `run_degrade` + `final_answer` | `@wrap_tool_call` (outside the retry) + `@after_agent` |
| a call count that varies | `max_steps` | `ModelCallLimitMiddleware(run_limit=6)` |

The framework makes the code shorter, but **it doesn't decide for us**: we still have to know which errors to retry, how to order the middleware, and what the user must see

**Exercises**
1. Set `ModelCallLimitMiddleware(run_limit=1)` and ask the same question — where does the agent stop, and what does it say?
2. Swap `cache_fallback` and `retry` in the list, then make search fail twice — how does the result change? Check the trace

**Next — the EP06 recap:** the six design principles and how to choose a pattern, from what we saw in the four labs